# SynBO Optimization Demo

This notebook demonstrates a complete **multi-objective reaction optimization** workflow
using SynBO on a cobalt-catalyzed asymmetric reaction example.

- **Reaction space**: 5 reagent types x 58,320 combinations
- **Objectives**: Maximize yield and ee


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from synbo import ReactionOptimizer
from synbo.utils import load_desc_dict, get_prev_rxn

sns.set_style('whitegrid')
%matplotlib inline

print('SynBO imported successfully')


## Step 1: Load the Example Reaction Space

The reaction space defines all possible reagent choices.
Each CSV file contains SMILES strings and reagent names.


In [ ]:
EXAMPLE_DIR = Path('.').resolve()

reagent_types = ['alkali', 'cobalt_catalyst', 'organo_catalyst', 'oxidant', 'solvent']

desc_dict, condition_dict = load_desc_dict(
    reagent_types=reagent_types,
    desc_dir=str(EXAMPLE_DIR / 'descriptors'),
    name_suffix='_RDKit',
    index_col='name',
    return_condition_dict=True,
)

print('Reaction Space Summary:')
print('-' * 50)
total = 1
for rtype, values in condition_dict.items():
    count = len(values)
    total *= count
    print(f'  {rtype:25s}: {count:3d} options')
print(f'  Total combinations: {total:,}')
print(f'  Descriptor dimensions: {sum(df.shape[1] for df in desc_dict.values())}')


## Step 2: Initial Sampling (Batch 0)

The first batch uses **Latin Hypercube Sampling (LHS)**
to generate a space-filling initial design.


In [ ]:
optimizer_init = ReactionOptimizer(
    opt_metrics=['yield', 'ee'],
    opt_type='init',
    random_seed=42,
    save_dir=str(EXAMPLE_DIR / 'results'),
)
optimizer_init.load_rxn_space(condition_dict)
optimizer_init.load_desc(desc_dict)
optimizer_init.initialize(batch_size=8, sampling_method='lhs')
optimizer_init.save_results(filetype='csv')

print('Initial batch recommended:')
for i, cond in enumerate(optimizer_init.selected_conditions):
    print(f'  {i+1}. {dict(zip(reagent_types, cond))}')


## Step 3: Simulate Experimental Results

In a real workflow, you would run experiments and fill in actual yield/ee values.
Here we simulate results for demonstration.


In [ ]:
batch0_file = list((EXAMPLE_DIR / 'results').glob('batch-0_*.csv'))[0]
batch0_data = pd.read_csv(batch0_file)

np.random.seed(42)
batch0_data['yield'] = np.random.uniform(30, 85, size=len(batch0_data))
batch0_data['ee'] = np.random.uniform(40, 95, size=len(batch0_data))

print(f'Simulated {len(batch0_data)} experimental results:')
display(batch0_data[['alkali', 'cobalt_catalyst', 'organo_catalyst', 'yield', 'ee']].round(1))


## Step 4: Bayesian Optimization (Batch 1)

Now we use the experimental results from Batch 0
to train a surrogate model and recommend the next batch.


In [ ]:
optimizer_opt = ReactionOptimizer(
    opt_metrics=['yield', 'ee'],
    opt_type='auto',
    random_seed=42,
    save_dir=str(EXAMPLE_DIR / 'results'),
)
optimizer_opt.load_rxn_space(condition_dict)
optimizer_opt.load_desc(desc_dict)
optimizer_opt.load_prev_rxn(batch0_data)
optimizer_opt.optimize(batch_size=5)
optimizer_opt.save_results(filetype='csv')

print('Batch 1 recommendations:')
for i, (cond, rtype, pred_mean, pred_std) in enumerate(zip(
    optimizer_opt.selected_conditions,
    optimizer_opt.recommend_type,
    optimizer_opt.pred_mean,
    optimizer_opt.pred_std
)):
    print(f'  {i+1}. [{rtype}] {dict(zip(reagent_types, cond))}')
    print(f'       Pred yield: {pred_mean[0]:.1f}+/-{pred_std[0]:.1f} | Pred ee: {pred_mean[1]:.1f}+/-{pred_std[1]:.1f}')

explore = sum(1 for t in optimizer_opt.recommend_type if t.lower() == 'explore')
exploit = sum(1 for t in optimizer_opt.recommend_type if t.lower() == 'exploit')
print(f'  Explore: {explore} | Exploit: {exploit}')


## Step 5: Simulate More Rounds

We simulate several rounds of optimization to show how the Pareto front evolves.


In [ ]:
all_data = batch0_data.copy()
n_rounds = 6

for round_num in range(1, n_rounds + 1):
    opt = ReactionOptimizer(opt_metrics=['yield', 'ee'], opt_type='auto',
        random_seed=42 + round_num, save_dir=str(EXAMPLE_DIR / 'results'))
    opt.load_rxn_space(condition_dict)
    opt.load_desc(desc_dict)
    opt.load_prev_rxn(all_data)
    opt.optimize(batch_size=5)
    
    new_data = pd.DataFrame(opt.selected_conditions, columns=reagent_types)
    new_data['batch'] = round_num
    new_data['yield'] = np.random.uniform(20 + round_num * 5, 95, size=len(new_data))
    new_data['ee'] = np.random.uniform(30 + round_num * 5, 98, size=len(new_data))
    all_data = pd.concat([all_data, new_data], ignore_index=True)
    print(f'Round {round_num}: +{len(new_data)} experiments, total: {len(all_data)}')

print(f'Simulated {len(all_data)} experiments across {n_rounds + 1} batches')


## Step 6: Visualize the Pareto Front

The **Pareto front** shows the trade-off between yield and ee.
Points on the front are non-dominated.


In [ ]:
def get_pareto_front(x, y):
    points = np.column_stack([x, y])
    is_pareto = np.ones(len(points), dtype=bool)
    for i in range(len(points)):
        for j in range(len(points)):
            if all(points[j] >= points[i]) and any(points[j] > points[i]):
                is_pareto[i] = False
                break
    return is_pareto

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

batch_ids = sorted(all_data['batch'].unique())
colors = plt.cm.viridis(np.linspace(0, 1, len(batch_ids)))

for bid, color in zip(batch_ids, colors):
    batch_data = all_data[all_data['batch'] == bid]
    axes[0].scatter(batch_data['yield'], batch_data['ee'],
        color=color, label=f'Batch {int(bid)}', alpha=0.7, s=60)

pareto_mask = get_pareto_front(all_data['yield'].values, all_data['ee'].values)
pareto_points = all_data[pareto_mask].sort_values('yield')
axes[0].plot(pareto_points['yield'], pareto_points['ee'],
    'r--', linewidth=2, label='Pareto Front')
axes[0].legend(fontsize=8)
axes[0].set_xlabel('Yield (%)')
axes[0].set_ylabel('ee (%)')
axes[0].set_title('Optimization Progress by Batch')

best_per_batch = all_data.groupby('batch').agg({'yield': 'max', 'ee': 'max'})
axes[1].plot(best_per_batch.index, best_per_batch['yield'], 'o-', label='Best Yield', linewidth=2)
axes[1].plot(best_per_batch.index, best_per_batch['ee'], 's-', label='Best ee', linewidth=2)
axes[1].set_xlabel('Batch')
axes[1].set_ylabel('Value (%)')
axes[1].set_title('Best Values Over Time')
axes[1].legend()

hv_values = []
for bid in batch_ids:
    batch_subset = all_data[all_data['batch'] <= bid]
    pmask = get_pareto_front(batch_subset['yield'].values, batch_subset['ee'].values)
    pareto = batch_subset[pmask]
    hv = np.prod(np.maximum(pareto[['yield', 'ee']].values / 100.0, 0).max(axis=0))
    hv_values.append(hv)

axes[2].bar(batch_ids, hv_values, color=plt.cm.viridis(np.linspace(0.1, 0.9, len(batch_ids))))
axes[2].set_xlabel('Batch')
axes[2].set_ylabel('Hypervolume')
axes[2].set_title('Hypervolume Progress')

plt.tight_layout()
plt.show()

print(f'Final Pareto front: {pareto_mask.sum()} points')
print(f'Best yield: {all_data["yield"].max():.1f}%')
print(f'Best ee: {all_data["ee"].max():.1f}%')


## Step 7: Final Pareto Front

Final visualization with the ideal point (100% yield, 100% ee).


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sc = ax.scatter(all_data['yield'], all_data['ee'],
    c=all_data['batch'], cmap='viridis', s=80, alpha=0.7, edgecolors='k', linewidth=0.5)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Batch')

pareto_mask = get_pareto_front(all_data['yield'].values, all_data['ee'].values)
pareto_points = all_data[pareto_mask].sort_values('yield')
ax.plot(pareto_points['yield'], pareto_points['ee'],
    'r-', linewidth=2.5, label='Pareto Front', zorder=5)
ax.scatter(pareto_points['yield'], pareto_points['ee'],
    color='red', s=120, zorder=6, edgecolors='darkred')

ax.scatter([100], [100], marker='*', color='gold', s=400,
    edgecolors='black', zorder=7, label='Ideal (100%, 100%)')

ax.set_xlabel('Yield (%)', fontsize=12)
ax.set_ylabel('ee (%)', fontsize=12)
ax.set_title('Pareto Front: Yield vs Enantiomeric Excess', fontsize=14)
ax.legend(loc='lower right')
ax.set_xlim(0, 105)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()


## Summary

This notebook demonstrated:

1. **Reaction Space Loading** - 5 reagent types, ~58k combinations
2. **Initial Sampling** - LHS generates an informative first batch
3. **Bayesian Optimization** - Multi-objective BO with explore/exploit trade-off
4. **Pareto Front Visualization** - Trade-off between yield and ee
5. **Hypervolume Tracking** - Quantitative measure of optimization progress

### Key Takeaways
- SynBO converges toward the Pareto front as more batches are evaluated
- Predictions include uncertainty estimates to help prioritize experiments
- Hypervolume increases as better trade-off points are discovered
- The framework supports both CLI and Python API workflows
